In [ ]:
!pip install pika



In [ ]:
import pika
import json

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"
RABBITMQ_VHOST = "/"

QUEUE_NAME = "student_wellness_predictions"

print("RabbitMQ configuration loaded")
print("Host:", RABBITMQ_HOST)
print("Port:", RABBITMQ_PORT)
print("Queue:", QUEUE_NAME)

RabbitMQ configuration loaded
Host: 129.153.75.221
Port: 5672
Queue: student_wellness_predictions


In [ ]:
from getpass import getpass

RABBITMQ_PASSWORD = getpass("Enter RabbitMQ password: ")

Enter RabbitMQ password: ··········


In [ ]:
credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

rabbitmq_parameters = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=credentials,
    heartbeat=60,
    blocked_connection_timeout=300
)

connection = pika.BlockingConnection(rabbitmq_parameters)
channel = connection.channel()

channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print("Connected to RabbitMQ successfully!")
print("Queue ready:", QUEUE_NAME)

Connected to RabbitMQ successfully!
Queue ready: student_wellness_predictions


In [ ]:
def process_message(message):
    student_id = message.get("student_id")
    predicted_risk = message.get("predicted_risk")
    probabilities = message.get("probabilities")

    print("\n========== MESSAGE PROCESSED ==========")
    print(f"Student ID     : {student_id}")
    print(f"Predicted Risk : {predicted_risk}")
    print(f"Probabilities  : {probabilities}")

    if predicted_risk == "High":
        print("Action         : Flag for early intervention review")

    elif predicted_risk == "Medium":
        print("Action         : Continue monitoring")

    else:
        print("Action         : No immediate action required")

    print("========================================")

In [ ]:
def callback(ch, method, properties, body):
    try:
        print("\nMessage received from RabbitMQ!")

        # Convert JSON message into Python dictionary
        message = json.loads(body)

        # Process the received message
        process_message(message)

        # Acknowledge successful processing
        ch.basic_ack(
            delivery_tag=method.delivery_tag
        )

        print("Message acknowledged successfully.")

    except Exception as e:
        print("Message processing failed:", e)

        # Reject the message without requeueing
        ch.basic_nack(
            delivery_tag=method.delivery_tag,
            requeue=False
        )

In [ ]:
print("Queue:", QUEUE_NAME)
print("Channel open:", channel.is_open)

method, properties, body = channel.basic_get(
    queue=QUEUE_NAME,
    auto_ack=False
)

if method:
    print("\nMessage found in queue!")
    print("Raw message:")
    print(body.decode())

    channel.basic_ack(
        delivery_tag=method.delivery_tag
    )

    print("\nMessage acknowledged.")
else:
    print("No message currently available in the queue.")

Queue: student_wellness_predictions
Channel open: True
No message currently available in the queue.


In [ ]:
channel.basic_qos(prefetch_count=1)

channel.basic_consume(
    queue=QUEUE_NAME,
    on_message_callback=callback,
    auto_ack=False
)

print("==========================================")
print("RabbitMQ Consumer Started")
print("Waiting for messages...")
print("Queue:", QUEUE_NAME)
print("==========================================")

channel.start_consuming()

RabbitMQ Consumer Started
Waiting for messages...
Queue: student_wellness_predictions

Message received from RabbitMQ!

========== MESSAGE PROCESSED ==========
Student ID     : 1001
Predicted Risk : Medium
Probabilities  : {'Low': 0.23, 'Medium': 70.53, 'High': 29.25}
Action         : Continue monitoring
Message acknowledged successfully.
